# Lecture 10 — `zip()`, `range()`, and `enumerate()`
### EES 3350/5350 · Python in Earth Science · Monday, September 21, 2026

**Learning objectives for today**
- quick refresher on list/dictionary methods (lectures 04 and 05)
- one new dictionary trick: unpacking with `**`
- pair up parallel lists with `zip()` instead of a position counter
- use `range()` for reliable, exact count-controlled loops
- use `enumerate()` to get index and value together

---
## Part 0 : set up

In [ ]:
import math

def factor_of_safety(cohesion, soil_depth, soil_density, 
                     slope_deg, friction_deg, rw,  gravity=9.81, water_density=1000):
    """
    Compute the infinite slope Factor of Safety (FS).

    Parameters
    ----------
    cohesion : float
        Cr + Cs : cohesion - bonds holding roots (r) and soil (s) together (Pa)
    soil_depth : float
        depth of soil perpendicular to the slope (m)
    soil_density : float
        density of wet soil (kg/m^3)
    gravity : float
        gravitational acceleration (m/s^2)
    slope_deg : float
        slope angle (angle of the soil surface from horizontal) (degrees)
    friction_deg : float
        internal friction angle of the soil (degrees)
    rw : float
        relative wetness - ratio of groundwater depth to soil depth (1 = fully saturated) (0-1)
    water_density : float
        density of water (kg/m^3)

    Returns
    -------
    float, or None if slope_deg == 0 (undefined, would divide by 0)
    """
    # converting degrees to radians
    theta = math.radians(slope_deg)
    phi = math.radians(friction_deg)

    # A
    denom_A = soil_depth * soil_density * gravity * math.sin(theta)
    A = cohesion/denom_A
    # B
    parenth_B = (1 - ((rw * water_density) / soil_density))
    numerat_B = math.cos(theta) * math.tan(phi) * parenth_B
    B = numerat_B / math.sin(theta)

    FS = A + B
    
    return FS

def interpret_stability(fs):
    """Classify a Factor of Safety value into a stability category."""
    if fs < 1.0:
        return "Unstable"
    elif fs < 1.3:
        return "Marginally stable"
    else:
        return "Stable"

In [ ]:
cohesion_values = [0, 4000, 8000, 12000, 26000, 20000]
fs_cohesion = []

for c in cohesion_values:
    fs = factor_of_safety(c, 1.0, 2000, 25, 32, 0.5)
    fs_cohesion.append(fs)

print(cohesion_values)
print(fs_cohesion)

---
## Part 1 : quick refresher on `list` and `dict` methods

You already know all of this from Lectures 04 and 05. 
#### A brief reminder of list methods
|**list method**| what it does | **dict method** | what it does |
|--:|:----|----:|:-|
| `.append(x)`| adds `x` to the end of the list | `.keys()`| returns all the keys | 
| `.sort()`| sorts the list in place | `.values()`| returns all the values | 
| `.index(x)`| gives you the position of the first `x` | `.items()`| key-value pairs together, as tuples | 
| `.insert(i, x)`| inserts `x` at position `i` | `.get(key, default)`| safely fetch a value; returns `default` instead of crashing if key is missing | 
| `.remove(x)`| removes the first item equal to `x` | `.update({...})`| add new key-value pairs, or overwrite existing ones | 
| `.pop(i)`| removes and returns the item at position `i` (last item if `i` is omitted) | `in` | checks membership
| `.copy()`| makes an independent copy of that list | 


Here's a short reminder that we can use these methods to modify lists, which is useful if data was logged in the wrong order like in cohesion_demo.

In [ ]:
cohesion_demo = cohesion_values.copy()
print("before       :", cohesion_demo)

cohesion_demo.sort()
print("after .sort():", cohesion_demo)

position = cohesion_demo.index(26000)
print(f"26000 is now at position {position}.")


scenarios = {
    "low":    {"cohesion": 0,     "soil_depth": 0.2, "soil_density": 2000,
               "slope_deg": 5,  "friction_deg": 25, "rw": 0.0},
    "medium": {"cohesion": 8500,  "soil_depth": 1.0, "soil_density": 2000,
               "slope_deg": 25, "friction_deg": 32, "rw": 0.5},
    "high":   {"cohesion": 17000, "soil_depth": 2.5, "soil_density": 2000,
               "slope_deg": 45, "friction_deg": 40, "rw": 1.0},
}

print (scenarios.keys())

We can then go into a dictionary like `scenarios` and easily run the `factor_of_safety` calculation in a single line with a `for` loop:

In [ ]:
for name, params in scenarios.items():
    fs = factor_of_safety(params["cohesion"], params["soil_depth"], params["soil_density"], params["slope_deg"], params["friction_deg"], params["rw"]) 
    print(f"{name:>6}: FS = {fs:.3f} -> {interpret_stability(fs)}")

#### New Dictionary Method: dictionary Unpacking with `**`
Typing out every `params["medium"]["cohesion"]` is tedious and `factor_of_safety()`'s parameter names already match `scenarios`'s inner keys **exactly**. When that is true, `**` lets you unpack an entire dictionary straight into a function call as keyword arguments.

In [ ]:
for name, params in scenarios.items():
    fs = factor_of_safety(**params) # unpack dictionary straight into the function
    print(f"{name:>6}: FS = {fs:.3f} -> {interpret_stability(fs)}")

<div class="alert alert-success">

**Tip**: aligning text with format specs

Inside an f-string, the `:` introduces a *format* spec. These are instructions for how to display a value. You've already seen something likke `{fs:.3f}` which displays the value rounded to 3 decimal places. Some new specs:

- `{name:>6}` right-aligns `name` in a 6 character-wide field, padding with spaces on the left if it's shorter
- `{name:<6}` left-aligns instead
- `{name:^6}` centers instead.

This is why `low`, `medium`, and `high` line up in a neat column above, with their colons stacked.
    
</div>

##### This gives you the same result, with a lot less typing, but it only works because every key in `params` matches a real parameter name in `factor_of_safety()`

<div class="alert alert-info">

### Exercise 1
Add a fourth scenario called `"extreme"` to `scenarios` with the proper method. Here are the details for that scenario:
- cohesion=20000
- soil_depth=3.0
- soil_density=2000
- slope_deg=40
- friction_deg=38
- rw=0.9

Then use `**` unpacking to print its FS and stability category, the same way the loop above does it for the other three.
    
</div>

In [ ]:
# your answer here
scenarios._______({"extreme": {______}})

fs = factor_of_safety(______)
print(f"extreme: FS = {___:.3f} -> {interpret_stability(fs)}")

---
## Part 2 : the `zip()` function

We have two lists that belong together: `cohesion_values` and `fs_cohesion`. With what you know so far, printing them side by side means a position counter:

In [ ]:
i = 0
while i < len(cohesion_values):
    print(f"cohesion={cohesion_values[i]} -> FS = {fs_cohesion[i]:.3f}")
    i += 1

That works, but you're doing two separate lookups (`cohesion_values[i]` and `fs_cohesion[i]`) just to walk two lists in step.

### `zip()` lets you walk through two (or more) lists together, pairing up items that share the same position. No counter required.


In [ ]:
for c, fs in zip(cohesion_values, fs_cohesion):
    print(f"cohesion={c} -> FS={fs:.3f}")

#### Let's walk through what just happened.
`zip(cohesion_values, fs_cohesion)` pairs up the two lists position by position:
|**position**|**`cohesion_values`**|**`fs_cohesion**|
|-|-|-|
|0|0|1.005|
|1|4000|1.487|
|2|8000|1.970|
|3|12000|2.452|
|4|26000|4.141|
|5|20000|3.417|

Each pass through the loop, `zip()` hands over one row of that table as a tuple: `(0, 1.005)` on the first pass, `(4000, 1.487)` on the second, and so on.

Writing `for c, fs in zip(...)` **unpacks** that tuple into two variables in one step:
- `c` takes the first item of the pair (the value from `cohesion_values`)
- `fs` takes the second (the matching value from `fs_cohesion`).

It's exactly like writing:
```python
for pair in zip(cohesion_values, fs_cohesion):
    c = pair[0]
    fs = pair[1]
    print(f"cohesion={c} -> FS={fs:.3f}")
```
Only, it's done for you on the `for` line, since unpacking a pair straight into two loop variables is such a common pattern.

`.zip()` isn't limited to two lists! Let's build a `categories` list, then zip all three together.

In [ ]:
categories = []

for fs in fs_cohesion:
    categories.append(interpret_stability(fs))

for c, fs, category in zip(cohesion_values, fs_cohesion, categories):
    print(f"cohesion={c:>6} | FS={fs:.3f} | {category}")

#### One more trick: combine `zip()` with the built-in `dict()` function
This will build a dictionary in a single line, no loop needed at all.

In [ ]:
stability_summary = dict(zip(cohesion_values, fs_cohesion))
print(stability_summary)

#### What just happened?

`dict(zip(cohesion_values, fs_cohesion)` is doing 2 things stacked together:
1. **`zip()` makes pairs** : `zip(cohesion_values, fs_cohesion)` walks through both lists at once and bundles each same-position pair into a tuple (see below). On its own, `zip()` doesn't make a dictionary, it just makes an object you can loop through, where each item is a 2-item tuple.
        - `(0, 1.005)`, `(4000, 1.487)`, `(8000, 1.970)`, and so on
2. **`dict()` turns pairs into key-value entries** : the built-in `dict` function can build a dictionary out of *any* sequence of 2-item pairs. It takes the first item of each pair as the key and the second as the value. So handing it the zipped pairs gives you:
```python
{0: 1.005, 4000: 1.487, 8000: 1.970, 12000: 2.452, 26000: 4.141, 20000: 3.417}
```

##### Another example. 
What if we have two lists (`names` and `values`) and we wanted to make them a dictionary? We could do the following:

In [ ]:
names = ["low", "medium", "high"]
values = [5.33, 2.03, 0.91]

dict(zip(names, values))

#### Common mistake: mismatched lengths
`zip()` will not tell you if your lists are different lengths. It just quietly stops at the shorter one...

In [ ]:
# suppose the last day's field notes got lost in transit
fs_cohesion_incomplete = fs_cohesion[:-1]

print("cohesion_values has", len(cohesion_values), "items")
print("fs_cohesion_incomplete has", len(fs_cohesion_incomplete), "items")
print()

for c, fs in zip(cohesion_values, fs_cohesion_incomplete):
    print(f"cohesion={c} -> FS={fs:.3f}")

**No error was raised.** Cohesion value `20000` never got printed, and nothing told you that happened. This is a case where the *absence* of an error is the dangerous part. It's a good habit to compare `len()` on both lists before you trust a `zip()`.

Let's set up a fresh sweep to practice on: the friction angle sweep from Lab 3, Exercise 9.

In [ ]:
friction_values = [15, 20, 25, 30, 35, 40, 45]
fs_friction = []

for f in friction_values:
    fs = factor_of_safety(8500, 1.0, 2000, 25, f, 0.5)
    fs_friction.append(fs)

print(friction_values)
print(fs_friction)

<div class="alert alert-info">

### Exercise 2
Using `zip()`, loop through `friction_values` and `fs_friction` together and print a sentence for each friction angle giving the angle, its FS, and its stability from `interpret_stability()`.

</div>

In [ ]:
# your answer here
for _____, _____ in zip(______, ______):
    print(f"______") 

<details>
<summary style="color:red;"><strong>Hint about `interpret_stability`.</strong></summary>
    
how did we use the interpret_stability function earlier in this lecture?

---
## Part 3 : `range()`
Remember last Wednesday's `while` loop that raised `rw` from `0.0` to `1.0` in steps of `0.1`? 

In [ ]:
rw = 0.0             # setting initial rw to dry
rw_increase = 0.1    # rw raises by this much each hour of the storm

while rw < 1.0:               # while statement that sets condition that as long as rw < 1.0...
    print(f"Rw = {rw:.1f}")   # prints Rw value rounded to 1 decimal place
    rw += rw_increase         # adds the increase (rw_increase) to the existing value of rw

print("Soil is fully saturated.") 

>>>> Side Note: Apparently `0.1` cannot be represented *exactly* in binary floating point (much like `1/3` can't be written exactly in decimal). Every time the loop adds `0.1`, tiny rounding errors creep in. By the 10th pass, `rw` is actually `0.9999999999999999` not `1.0`.
>>>> 
>>>> The `while rw < 1.0` check compares the *real* underlying value, so `0.9999999999999999 < 1.0` is `True` and the loop runs one more time.
>>>> 
>>>> **The fix?** Don't build up a decimal by repeated addition! Count in exact integers with `range()`, and only convert to a decimal fresh each time. 

That was a `while` loop because at the time, you didn't have another tool for "do this a set number of times." 

### Let's use `range()` to do something 11 times:

In [ ]:
for i in range(11):
    rw = i / 10 
    print(f"Rw = {rw:.1f}")
print("Soil is fully saturated.")

`range(11)` produces the whole numbers `0` through `10`:
|**what you write**|**what it produces**|
|---:|:---|
|`range(11)`| `0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10`|

That's 11 numbers total, stopping *before* 11. The count always matches the number you gave it. `i` is a clean integer every single pass (never an accumulated decimal), so dividing each one by `10` each time gives you `0.0, 0.1, 0.2, ..., 1.0` with no drift at all.

`range()` can also take a **start**, **stop**, and **step**, instead of counting from 0 by 1s. Recall Lab 03, Exercise 6: the slope sweep from 5$^\circ$ to 45$^\circ$ in steps of 5$^\circ$. Here it is with `range()`, at the medium cohesion/friction values under saturated conditions:

In [ ]:
for slope in range(5, 50, 5):
    fs = factor_of_safety(8500, 1.0, 2000, slope, 32, 1.0)
    print(f"slope={slope:>2} deg -> FS ={fs:.3f} -> {interpret_stability(fs)}")

|**what you write**|**what it produces**|
|---:|:---|
|`range(5, 50, 5)`| `5, 10, 15, 20, 25, 30, 35, 40, 45`|

`range(start, stop, step)` counts from `start` up to (but not including; so its exclusive) `stop`, in increments of `step`. We wrote `50`, not `45`, as the stop because `range()` will always stop *before* its stop value. So `range(5, 45, 5)` would have cut off at `40` and silently dropped `45` from the sweep.

<div class="alert alert-info">

### Exercise 3
Recall the relative wetness sweep from Lab 3, Exercise 10: Rw values `0.0, 0.2, 0.4, 0.6, 0.8, 1.0` (cohesion=2000, soil_depth=1.0, slope_deg=32, friction_deg=30).

Using `range()` with a start, stop, and step (**not** a decimal step, since `range()` only works with whole numbers) compute and print FS for each Rw value.
    
</div>

<details>
<summary style="color:red;"><strong>Hint.</strong></summary>
    
count in whole numbers that are 10x the Rw values you want then divide by 10 to get the actual Rw values. This is the same trick we just used to fix the saturation loop).

In [ ]:
# your answer here
for step in range(_____, _____, _____):
    rw = _____
    fs = factor_of_safety(2000, 1.0, 2000, 32, 30, rw)
    print(f"Rw={rw:.1f} -> FS={fs:.3f}")

---
## Part 4: `enumerate()`
`for` loops so far have handed you *values* one at a time. `enumerate()` hands you the **index and the value together**, as a pair.

### Let's use `enumerate()` to easily go through `fs_cohesion`

In [ ]:
for i, fs in enumerate(fs_cohesion):
    print(f"step {i}: FS = {fs:.3f}")

| **what you write** | **what each pass produces** |
|---|---|
| `enumerate(fs_cohesion)` | `(0, 1.005)`, `(1, 1.487)`, `(2, 1.970)`, `(3, 2.452)`, `(4, 4.141)`, `(5, 3.417)` |

Just like `zip()`, `for i, fs in enumerate(...)` unpacks each pair into two variables:
- `i` (the first variable listed) gets the position 
- `fs` (the second variable listed) gets the value at that position

And since `enumerate()` and `zip()` both hand you tuples, you can combine them:

In [ ]:
for i, (c, fs) in enumerate(zip(cohesion_values, fs_cohesion)):
    print(f"step {i}: cohesion={c} -> FS ={fs:.3f}")

<div class="alert alert-info">

### Exercise 4
Recall the soil depth sweep from Lab 3, Exercise 11: `depth_values = 0.25, 0.5, 1.0, 1.5, 2.0, 3.0` (cohesion=8500, slope_deg=25, friction_deg=32, rw=0.5).

Using `enumerate()` directly on `depth_values` (**not** `range(len(...))`) print the step number, the soil depth, its FS, and its stability category for each.
    
</div>

In [ ]:
# your answer here
depth_values = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]

for _____, _____ in enumerate(_____):
    fs = factor_of_safety(_____)
    print(f"step {i}: depth={depth} m -> FS={fs:.3f} -> {interpret_stability(fs)}")

---
## Practice

<div class="alert alert-info">

### Exercise 5

Atmospheric pressure drops as elevation increases: it starts at 101.325 kPa at sea level and decreases by 1.2 kPa for every 100 m of elevation gain. Using a count-controlled `for` loop with `range()`, calculate and print pressure from 500 m to 5000 m elevation, in 500 m increments, along with a warning message:
- "Extreme altitude conditions" if pressure < 50 kPa
- "High altitude effects likely" if pressure is 50-70 kPa
- "Normal conditions" otherwise
    
</div>

In [ ]:
# your answer here
sea_level_pressure = 101.325
decrease_per_100m = 1.2

# for loop



<div class="alert alert-info">

### Exercise 6
A seismometer recorded these earthquake magnitudes over a week: `earthquake_magnitudes = [2.1, 4.5, 1.8, 3.2, 5.1, 2.9, 3.7]`. Using `enumerate()`, print a reading number (starting at 1, not 0), the magnitude, and its classification.

In [ ]:
# your answer here
earthquake_magnitudes = [2.1, 4.5, 1.8, 3.2, 5.1, 2.9, 3.7]

# for loop


